# Experimentos de modelagem

Notebook independente da API para experimentar os modelos candidatos com `dados_tratados/dados_tratados/dados_para_teste.csv`.

Aqui ficam as listas de features, preprocessamento, metricas, busca de hiperparametros e treino final. Assim voce pode testar mudancas livremente antes de levar algo para a pipeline da API.

## 1. Imports e configuracoes

In [60]:
%pip install xgboost lightgbm catboost

Defaulting to user installation because normal site-packages is not writeable
  Using cached xgboost-3.3.0-py3-none-win_amd64.whl.metadata (2.0 kB)
  Using cached narwhals-2.22.1-py3-none-any.whl.metadata (15 kB)
Using cached xgboost-3.3.0-py3-none-win_amd64.whl (69.5 MB)
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 1.5/1.5 MB 10.9 MB/s  0:00:00
   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   - -------------------------------------- 4.2/100.2 MB 21.0 MB/s eta 0:00:05
   ---- ----------------------------------- 10.2/100.2 MB 25.5 MB/s eta 0:00:04
   ------ --------------------------------- 17.0/100.2 MB 28.3 MB/s eta 0:00:03
   --------- ------------------------------ 24.9/100.2 MB 31.0 MB/s eta 0:00:03
   ------------- -------------------------- 34.6/100.2 MB 33.8 MB/s eta 0:00:02
   ---------------- ----------------------- 40.1/100.2 MB 32.7 MB/s eta 0:00:02
   ------------------ ------------


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\andriel_orbi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [41]:
!pip install optuna

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\andriel_orbi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [42]:
from pathlib import Path
from typing import Any, Callable

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVR
import optuna
try:
    import optuna
except ImportError:
    optuna = None

RAIZ_PROJETO = Path.cwd()
if RAIZ_PROJETO.name == "modelos de aprendizagem":
    RAIZ_PROJETO = RAIZ_PROJETO.parent

DADOS_PATH = RAIZ_PROJETO / "dados_tratados" / "dados_tratados" / "dados_para_teste.csv"
ARTIFACTS_DIR = RAIZ_PROJETO / "modelos de aprendizagem" / "artifacts_modelagem"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
N_SPLITS = 3
N_TRIALS = 10
TEST_SIZE = 0.15

## 2. Features e alvo

In [43]:
TARGET = "preco"

NUMERIC_FEATURES = [
    "area_m2",
    "quartos",
    "banheiros",
    "suites",
    "andar",
    "vagas",
]

BOOLEAN_FEATURES = [
    "portaria",
    "vista_mar",
    "condominio_fechado",
    "piscina",
    "deck",
    "varanda_gourmet",
    "varanda",
    "academia",
    "salao_festa",
    "salao_jogos",
    "quadra_campo",
]

CATEGORICAL_FEATURES = [
    "bairro",
    "tipo_imovel_padronizado",
]

MODEL_FEATURES = CATEGORICAL_FEATURES + NUMERIC_FEATURES + BOOLEAN_FEATURES

## 3. Carregamento dos dados

In [44]:
df = pd.read_csv(DADOS_PATH)

faltantes = [coluna for coluna in MODEL_FEATURES + [TARGET] if coluna not in df.columns]
if faltantes:
    raise ValueError(f"Colunas obrigatorias ausentes: {faltantes}")

dados = df[MODEL_FEATURES + [TARGET]].copy()

for coluna in NUMERIC_FEATURES + [TARGET]:
    dados[coluna] = pd.to_numeric(dados[coluna], errors="coerce")

for coluna in BOOLEAN_FEATURES:
    dados[coluna] = dados[coluna].fillna(False).astype(bool)

dados = dados.dropna(subset=MODEL_FEATURES + [TARGET]).reset_index(drop=True)
X = dados[MODEL_FEATURES].copy()
y = dados[TARGET].copy()

print(f"Dataset: {DADOS_PATH}")
print(f"Linhas validas: {len(X)}")
print(f"Colunas: {len(X.columns)} features + alvo")
display(X.head())

Dataset: c:\Users\andriel_orbi\Tiago\git\pipeline-de-modelos-preditores-de-precos-de-imoveis-com-dados-em-tempo-real\dados_tratados\dados_tratados\dados_para_teste.csv
Linhas validas: 824
Colunas: 19 features + alvo


,bairro,tipo_imovel_padronizado,area_m2,quartos,banheiros,suites,andar,vagas,portaria,vista_mar,condominio_fechado,piscina,deck,varanda_gourmet,varanda,academia,salao_festa,salao_jogos,quadra_campo
0,Cambeba,apartamento_padrao,54.0,2.0,2.0,1.0,1.0,1.0,True,False,True,True,True,False,False,True,True,False,True
1,Mondubim,casa_padrao,100.0,3.0,3.0,2.0,0.0,2.0,False,False,False,False,True,False,True,False,False,False,False
2,Aracapé,apartamento_padrao,51.0,2.0,2.0,1.0,1.0,1.0,True,False,True,True,False,True,True,False,True,False,True
3,Aldeota,apartamento_padrao,135.0,3.0,4.0,3.0,0.0,3.0,False,False,False,True,False,False,False,True,True,True,True
4,Mondubim,apartamento_padrao,63.0,3.0,2.0,1.0,2.0,2.0,False,False,False,True,False,False,True,False,False,False,False


## 4. Preprocessamento e metricas

In [45]:
def one_hot_encoder() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def criar_preprocessador() -> ColumnTransformer:
    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", one_hot_encoder()),
        ]
    )
    return ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, NUMERIC_FEATURES),
            ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
            ("boolean", "passthrough", BOOLEAN_FEATURES),
        ],
        remainder="drop",
    )


def rmse(y_true: pd.Series | np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def calcular_metricas(y_true: pd.Series | np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    return {
        "rmse": rmse(y_true, y_pred),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
    }


def criar_pipeline(estimator: Any) -> Pipeline:
    return Pipeline(
        steps=[
            ("preprocessor", criar_preprocessador()),
            ("regressor", estimator),
        ]
    )

## 5. Modelos candidatos

In [46]:
def require_xgboost():
    try:
        from xgboost import XGBRegressor
    except ImportError as exc:
        raise ImportError("Instale xgboost para rodar este experimento.") from exc
    return XGBRegressor


def require_lightgbm():
    try:
        from lightgbm import LGBMRegressor
    except ImportError as exc:
        raise ImportError("Instale lightgbm para rodar este experimento.") from exc
    return LGBMRegressor


def require_catboost():
    try:
        from catboost import CatBoostRegressor
    except ImportError as exc:
        raise ImportError("Instale catboost para rodar este experimento.") from exc
    return CatBoostRegressor


def build_xgboost(params: dict[str, Any]):
    XGBRegressor = require_xgboost()
    return XGBRegressor(
        objective="reg:squarederror",
        eval_metric="rmse",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=0,
        **params,
    )


def build_lightgbm(params: dict[str, Any]):
    LGBMRegressor = require_lightgbm()
    return LGBMRegressor(
        objective="regression",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1,
        **params,
    )


def build_catboost(params: dict[str, Any]):
    CatBoostRegressor = require_catboost()
    return CatBoostRegressor(
        loss_function="RMSE",
        eval_metric="RMSE",
        random_seed=RANDOM_STATE,
        verbose=False,
        allow_writing_files=False,
        **params,
    )


def build_random_forest(params: dict[str, Any]):
    return RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1, **params)


def build_ridge(params: dict[str, Any]):
    return Ridge(random_state=RANDOM_STATE, **params)


def build_lasso(params: dict[str, Any]):
    return Lasso(random_state=RANDOM_STATE, max_iter=10_000, **params)


def build_svr(params: dict[str, Any]):
    return SVR(**params)


def build_mlp(params: dict[str, Any]):
    params = dict(params)
    if "hidden_layer_sizes" not in params:
        width = params.pop("width", 128)
        depth = params.pop("depth", 1)
        params["hidden_layer_sizes"] = tuple([width] * depth)
    else:
        params.pop("width", None)
        params.pop("depth", None)

    return MLPRegressor(
        random_state=RANDOM_STATE,
        early_stopping=True,
        max_iter=700,
        **params,
    )

## 6. Espacos de busca

In [47]:
def suggest_xgboost(trial):
    return {
        "n_estimators": trial.suggest_int("n_estimators", 300, 1500),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.20, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 10.0),
        "subsample": trial.suggest_float("subsample", 0.60, 1.00),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.60, 1.00),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
    }


def suggest_lightgbm(trial):
    return {
        "n_estimators": trial.suggest_int("n_estimators", 300, 1500),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.20, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 16, 256),
        "max_depth": trial.suggest_int("max_depth", 3, 14),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
        "subsample": trial.suggest_float("subsample", 0.60, 1.00),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.60, 1.00),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
    }


def suggest_catboost(trial):
    return {
        "iterations": trial.suggest_int("iterations", 300, 1500),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.20, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 20.0, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "random_strength": trial.suggest_float("random_strength", 0.0, 2.0),
    }


def suggest_random_forest(trial):
    return {
        "n_estimators": trial.suggest_int("n_estimators", 200, 900, step=100),
        "max_depth": trial.suggest_int("max_depth", 4, 32),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", 1.0]),
    }


def suggest_svr(trial):
    return {
        "kernel": trial.suggest_categorical("kernel", ["rbf", "poly"]),
        "C": trial.suggest_float("C", 1.0, 500.0, log=True),
        "epsilon": trial.suggest_float("epsilon", 0.001, 1.0, log=True),
        "gamma": trial.suggest_categorical("gamma", ["scale", "auto"]),
    }


def suggest_mlp(trial):
    width = trial.suggest_categorical("width", [64, 128, 256])
    depth = trial.suggest_int("depth", 1, 3)
    return {
        "hidden_layer_sizes": tuple([width] * depth),
        "activation": trial.suggest_categorical("activation", ["relu", "tanh"]),
        "alpha": trial.suggest_float("alpha", 1e-6, 1e-2, log=True),
        "learning_rate_init": trial.suggest_float("learning_rate_init", 1e-4, 1e-2, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [32, 64, 128]),
    }


MODEL_SPECS = {
    "xgboost": {"nome": "XGBoost", "builder": build_xgboost, "strategy": "optuna", "suggest": suggest_xgboost},
    "lightgbm": {"nome": "LightGBM", "builder": build_lightgbm, "strategy": "optuna", "suggest": suggest_lightgbm},
    "catboost": {"nome": "CatBoost", "builder": build_catboost, "strategy": "optuna", "suggest": suggest_catboost},
    "random_forest": {"nome": "Random Forest", "builder": build_random_forest, "strategy": "optuna", "suggest": suggest_random_forest},
    "ridge": {"nome": "Ridge", "builder": build_ridge, "strategy": "grid", "grid": {"alpha": [0.01, 0.1, 1.0, 10.0, 100.0]}},
    "lasso": {"nome": "Lasso", "builder": build_lasso, "strategy": "grid", "grid": {"alpha": [0.001, 0.01, 0.1, 1.0, 10.0]}},
    "svr": {"nome": "SVR", "builder": build_svr, "strategy": "optuna", "suggest": suggest_svr},
    "mlp": {"nome": "MLPRegressor", "builder": build_mlp, "strategy": "optuna", "suggest": suggest_mlp},
}

list(MODEL_SPECS)

['xgboost',
 'lightgbm',
 'catboost',
 'random_forest',
 'ridge',
 'lasso',
 'svr',
 'mlp']

## 7. Rotinas de treino

In [48]:
def cross_validate_params(builder: Callable[[dict[str, Any]], Any], params: dict[str, Any]) -> dict[str, float]:
    cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    fold_metrics = []

    for train_idx, valid_idx in cv.split(X, y):
        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
        pipeline = criar_pipeline(builder(params))
        pipeline.fit(X_train, y_train)
        pred = pipeline.predict(X_valid)
        fold_metrics.append(calcular_metricas(y_valid, pred))

    return {
        "rmse": float(np.mean([m["rmse"] for m in fold_metrics])),
        "mae": float(np.mean([m["mae"] for m in fold_metrics])),
        "r2": float(np.mean([m["r2"] for m in fold_metrics])),
    }


def buscar_com_grid(model_key: str, spec: dict[str, Any]) -> tuple[dict[str, Any], dict[str, float]]:
    pipeline = criar_pipeline(spec["builder"]({}))
    grid = {f"regressor__{key}": value for key, value in spec["grid"].items()}
    cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    search = GridSearchCV(
        pipeline,
        param_grid=grid,
        scoring="neg_root_mean_squared_error",
        cv=cv,
        n_jobs=-1,
        refit=True,
    )
    search.fit(X, y)
    best_params = {key.replace("regressor__", ""): value for key, value in search.best_params_.items()}
    best_metrics = cross_validate_params(spec["builder"], best_params)
    return best_params, best_metrics


def buscar_com_optuna(model_key: str, spec: dict[str, Any]) -> tuple[dict[str, Any], dict[str, float]]:
    if optuna is None:
        raise ImportError("Instale optuna para usar busca bayesiana.")

    def objective(trial):
        params = spec["suggest"](trial)
        metrics = cross_validate_params(spec["builder"], params)
        trial.set_user_attr("mae", metrics["mae"])
        trial.set_user_attr("r2", metrics["r2"])
        return metrics["rmse"]

    study = optuna.create_study(direction="minimize", study_name=f"{model_key}_rmse")
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)
    best_params = dict(study.best_params)
    best_metrics = cross_validate_params(spec["builder"], best_params)
    return best_params, best_metrics


def ajustar_modelo_final(builder: Callable[[dict[str, Any]], Any], params: dict[str, Any]) -> Pipeline:
    pipeline = criar_pipeline(builder(params))
    pipeline.fit(X, y)
    return pipeline


rankings = {}


def treinar_modelo(model_key: str) -> pd.DataFrame:
    if model_key not in MODEL_SPECS:
        raise KeyError(f"Modelo desconhecido: {model_key}")

    spec = MODEL_SPECS[model_key]
    print(f"\n=== Treinando {spec['nome']} ===")

    try:
        if spec["strategy"] == "grid":
            best_params, best_metrics = buscar_com_grid(model_key, spec)
        else:
            best_params, best_metrics = buscar_com_optuna(model_key, spec)
    except ImportError as exc:
        print(f"Pulando {spec['nome']}: {exc}")
        return pd.DataFrame()

    estimator = ajustar_modelo_final(spec["builder"], best_params)
    model_dir = ARTIFACTS_DIR / model_key
    model_dir.mkdir(parents=True, exist_ok=True)
    joblib.dump(estimator, model_dir / f"{model_key}.pkl")

    ranking = pd.DataFrame(
        [
            {
                "model_key": model_key,
                "modelo": spec["nome"],
                "rmse": best_metrics["rmse"],
                "mae": best_metrics["mae"],
                "r2": best_metrics["r2"],
                "best_params": best_params,
            }
        ]
    )
    ranking.to_csv(model_dir / "ranking.csv", index=False)
    rankings[model_key] = ranking

    print(
        f"{spec['nome']}: RMSE={best_metrics['rmse']:.2f} | "
        f"MAE={best_metrics['mae']:.2f} | R2={best_metrics['r2']:.4f}"
    )
    print(f"Modelo salvo em: {model_dir / f'{model_key}.pkl'}")
    display(ranking)
    return ranking


def ranking_geral() -> pd.DataFrame:
    if not rankings:
        return pd.DataFrame()
    return pd.concat(rankings.values(), ignore_index=True).sort_values("rmse", ignore_index=True)

## 8. Ridge

In [49]:
ranking_ridge = treinar_modelo("ridge")


=== Treinando Ridge ===
Ridge: RMSE=522598.97 | MAE=320852.33 | R2=0.5414
Modelo salvo em: c:\Users\andriel_orbi\Tiago\git\pipeline-de-modelos-preditores-de-precos-de-imoveis-com-dados-em-tempo-real\modelos de aprendizagem\artifacts_modelagem\ridge\ridge.pkl


,model_key,modelo,rmse,mae,r2,best_params
0,ridge,Ridge,522598.97262,320852.327578,0.541375,{'alpha': 10.0}


## 9. Lasso

In [50]:
ranking_lasso = treinar_modelo("lasso")


=== Treinando Lasso ===
Lasso: RMSE=544567.89 | MAE=348540.65 | R2=0.5037
Modelo salvo em: c:\Users\andriel_orbi\Tiago\git\pipeline-de-modelos-preditores-de-precos-de-imoveis-com-dados-em-tempo-real\modelos de aprendizagem\artifacts_modelagem\lasso\lasso.pkl


,model_key,modelo,rmse,mae,r2,best_params
0,lasso,Lasso,544567.887442,348540.647732,0.503654,{'alpha': 10.0}


## 10. SVR

In [51]:
ranking_svr = treinar_modelo("svr")

[I 2026-06-22 17:23:20,754] A new study created in memory with name: svr_rmse
[I 2026-06-22 17:23:20,907] Trial 0 finished with value: 804667.0695476602 and parameters: {'kernel': 'rbf', 'C': 11.932781285083616, 'epsilon': 0.03707037674631572, 'gamma': 'auto'}. Best is trial 0 with value: 804667.0695476602.



=== Treinando SVR ===


[I 2026-06-22 17:23:21,008] Trial 1 finished with value: 804800.7865705151 and parameters: {'kernel': 'poly', 'C': 1.233120527687171, 'epsilon': 0.03317197835232713, 'gamma': 'auto'}. Best is trial 0 with value: 804667.0695476602.
[I 2026-06-22 17:23:21,157] Trial 2 finished with value: 802595.0475881441 and parameters: {'kernel': 'rbf', 'C': 200.30100040877755, 'epsilon': 0.004115949259301723, 'gamma': 'auto'}. Best is trial 2 with value: 802595.0475881441.
[I 2026-06-22 17:23:21,260] Trial 3 finished with value: 804800.74074921 and parameters: {'kernel': 'poly', 'C': 1.6706845925454146, 'epsilon': 0.002227657486727839, 'gamma': 'auto'}. Best is trial 2 with value: 802595.0475881441.
[I 2026-06-22 17:23:21,457] Trial 4 finished with value: 803488.9260720238 and parameters: {'kernel': 'rbf', 'C': 53.604017871362245, 'epsilon': 0.05385415668618935, 'gamma': 'scale'}. Best is trial 2 with value: 802595.0475881441.
[I 2026-06-22 17:23:21,653] Trial 5 finished with value: 804473.8663553757

SVR: RMSE=801496.35 | MAE=492965.01 | R2=-0.0729
Modelo salvo em: c:\Users\andriel_orbi\Tiago\git\pipeline-de-modelos-preditores-de-precos-de-imoveis-com-dados-em-tempo-real\modelos de aprendizagem\artifacts_modelagem\svr\svr.pkl


,model_key,modelo,rmse,mae,r2,best_params
0,svr,SVR,801496.352402,492965.012962,-0.07287,"{'kernel': 'poly', 'C': 73.58174688831143, 'ep..."


## 11. MLPRegressor

In [52]:
ranking_mlp = treinar_modelo("mlp")

[I 2026-06-22 17:23:22,346] A new study created in memory with name: mlp_rmse



=== Treinando MLPRegressor ===


[I 2026-06-22 17:23:22,802] Trial 0 finished with value: 1268214.4184119445 and parameters: {'width': 64, 'depth': 3, 'activation': 'tanh', 'alpha': 0.008389152369752302, 'learning_rate_init': 0.00014992383643164535, 'batch_size': 32}. Best is trial 0 with value: 1268214.4184119445.
[I 2026-06-22 17:23:24,019] Trial 1 finished with value: 1268101.1975261755 and parameters: {'width': 128, 'depth': 3, 'activation': 'tanh', 'alpha': 8.818124502907882e-05, 'learning_rate_init': 0.005566982511402886, 'batch_size': 32}. Best is trial 1 with value: 1268101.1975261755.
[I 2026-06-22 17:23:24,237] Trial 2 finished with value: 1268209.5975248374 and parameters: {'width': 64, 'depth': 2, 'activation': 'tanh', 'alpha': 5.1925145755455e-06, 'learning_rate_init': 0.002281962342954997, 'batch_size': 128}. Best is trial 1 with value: 1268101.1975261755.
[I 2026-06-22 17:23:25,146] Trial 3 finished with value: 1268082.1956075123 and parameters: {'width': 128, 'depth': 2, 'activation': 'tanh', 'alpha': 

MLPRegressor: RMSE=525172.50 | MAE=325875.00 | R2=0.5362
Modelo salvo em: c:\Users\andriel_orbi\Tiago\git\pipeline-de-modelos-preditores-de-precos-de-imoveis-com-dados-em-tempo-real\modelos de aprendizagem\artifacts_modelagem\mlp\mlp.pkl


,model_key,modelo,rmse,mae,r2,best_params
0,mlp,MLPRegressor,525172.499782,325874.998555,0.536178,"{'width': 128, 'depth': 3, 'activation': 'relu..."


## 12. Random Forest

In [53]:
ranking_random_forest = treinar_modelo("random_forest")

[I 2026-06-22 17:23:58,553] A new study created in memory with name: random_forest_rmse



=== Treinando Random Forest ===


[I 2026-06-22 17:24:00,786] Trial 0 finished with value: 519017.9382585298 and parameters: {'n_estimators': 500, 'max_depth': 7, 'min_samples_split': 15, 'min_samples_leaf': 4, 'max_features': 'sqrt'}. Best is trial 0 with value: 519017.9382585298.
[I 2026-06-22 17:24:03,184] Trial 1 finished with value: 614420.3833466937 and parameters: {'n_estimators': 600, 'max_depth': 12, 'min_samples_split': 6, 'min_samples_leaf': 9, 'max_features': 'log2'}. Best is trial 0 with value: 519017.9382585298.
[I 2026-06-22 17:24:04,356] Trial 2 finished with value: 487060.81986383797 and parameters: {'n_estimators': 300, 'max_depth': 30, 'min_samples_split': 11, 'min_samples_leaf': 1, 'max_features': 'log2'}. Best is trial 2 with value: 487060.81986383797.
[I 2026-06-22 17:24:05,315] Trial 3 finished with value: 575508.7094298182 and parameters: {'n_estimators': 200, 'max_depth': 9, 'min_samples_split': 18, 'min_samples_leaf': 5, 'max_features': 'log2'}. Best is trial 2 with value: 487060.81986383797.


Random Forest: RMSE=469710.85 | MAE=280957.25 | R2=0.6270
Modelo salvo em: c:\Users\andriel_orbi\Tiago\git\pipeline-de-modelos-preditores-de-precos-de-imoveis-com-dados-em-tempo-real\modelos de aprendizagem\artifacts_modelagem\random_forest\random_forest.pkl


,model_key,modelo,rmse,mae,r2,best_params
0,random_forest,Random Forest,469710.853274,280957.252317,0.627016,"{'n_estimators': 300, 'max_depth': 16, 'min_sa..."


## 13. XGBoost

In [61]:


ranking_xgboost = treinar_modelo("xgboost")

[I 2026-06-22 17:40:18,400] A new study created in memory with name: xgboost_rmse



=== Treinando XGBoost ===


[I 2026-06-22 17:40:19,676] Trial 0 finished with value: 479466.1922852465 and parameters: {'n_estimators': 304, 'learning_rate': 0.06971133272108461, 'max_depth': 7, 'min_child_weight': 5.24864996420774, 'subsample': 0.6654673469223227, 'colsample_bytree': 0.8739322818123312, 'reg_alpha': 0.00024248592526268932, 'reg_lambda': 0.9598521345946178}. Best is trial 0 with value: 479466.1922852465.
[I 2026-06-22 17:40:23,188] Trial 1 finished with value: 492220.07721392764 and parameters: {'n_estimators': 1417, 'learning_rate': 0.1959030985066937, 'max_depth': 4, 'min_child_weight': 1.7970663796645665, 'subsample': 0.824066819781592, 'colsample_bytree': 0.8423538234518411, 'reg_alpha': 1.350955245082087e-07, 'reg_lambda': 0.9322978353369031}. Best is trial 0 with value: 479466.1922852465.
[I 2026-06-22 17:40:28,000] Trial 2 finished with value: 479982.8782572206 and parameters: {'n_estimators': 1446, 'learning_rate': 0.022832687630383288, 'max_depth': 7, 'min_child_weight': 8.19847789105983

XGBoost: RMSE=458521.72 | MAE=255767.14 | R2=0.6482
Modelo salvo em: c:\Users\andriel_orbi\Tiago\git\pipeline-de-modelos-preditores-de-precos-de-imoveis-com-dados-em-tempo-real\modelos de aprendizagem\artifacts_modelagem\xgboost\xgboost.pkl


,model_key,modelo,rmse,mae,r2,best_params
0,xgboost,XGBoost,458521.719595,255767.143094,0.6482,"{'n_estimators': 703, 'learning_rate': 0.01279..."


## 14. LightGBM

In [62]:
ranking_lightgbm = treinar_modelo("lightgbm")

[I 2026-06-22 17:40:59,017] A new study created in memory with name: lightgbm_rmse



=== Treinando LightGBM ===


C:\Users\andriel_orbi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\andriel_orbi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\andriel_orbi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
[I 2026-06-22 17:41:02,603] Trial 0 finished with value: 474406.55103531136 and parameters: {'n_estimators': 1332, 

LightGBM: RMSE=471870.13 | MAE=287913.52 | R2=0.6264
Modelo salvo em: c:\Users\andriel_orbi\Tiago\git\pipeline-de-modelos-preditores-de-precos-de-imoveis-com-dados-em-tempo-real\modelos de aprendizagem\artifacts_modelagem\lightgbm\lightgbm.pkl


,model_key,modelo,rmse,mae,r2,best_params
0,lightgbm,LightGBM,471870.128507,287913.522215,0.626413,"{'n_estimators': 1444, 'learning_rate': 0.0134..."


## 15. CatBoost

In [63]:
ranking_catboost = treinar_modelo("catboost")

[I 2026-06-22 17:41:16,021] A new study created in memory with name: catboost_rmse



=== Treinando CatBoost ===


[I 2026-06-22 17:42:11,631] Trial 0 finished with value: 494490.1708051761 and parameters: {'iterations': 590, 'learning_rate': 0.020663375289911964, 'depth': 10, 'l2_leaf_reg': 7.970318252910788, 'bagging_temperature': 0.055192856405121504, 'random_strength': 0.06379530066248695}. Best is trial 0 with value: 494490.1708051761.
[I 2026-06-22 17:42:14,632] Trial 1 finished with value: 454318.2311339816 and parameters: {'iterations': 1093, 'learning_rate': 0.014487286737428693, 'depth': 4, 'l2_leaf_reg': 2.5967779848542487, 'bagging_temperature': 0.08054969686717606, 'random_strength': 0.8546600896019456}. Best is trial 1 with value: 454318.2311339816.
[I 2026-06-22 17:42:15,765] Trial 2 finished with value: 449257.118648838 and parameters: {'iterations': 354, 'learning_rate': 0.04781576821983989, 'depth': 4, 'l2_leaf_reg': 1.1030213648531075, 'bagging_temperature': 0.8560483743196365, 'random_strength': 0.8166336337139621}. Best is trial 2 with value: 449257.118648838.
[I 2026-06-22 17:

CatBoost: RMSE=438707.39 | MAE=251630.09 | R2=0.6758
Modelo salvo em: c:\Users\andriel_orbi\Tiago\git\pipeline-de-modelos-preditores-de-precos-de-imoveis-com-dados-em-tempo-real\modelos de aprendizagem\artifacts_modelagem\catboost\catboost.pkl


,model_key,modelo,rmse,mae,r2,best_params
0,catboost,CatBoost,438707.387901,251630.093751,0.675836,"{'iterations': 1392, 'learning_rate': 0.173733..."


## 16. Ranking consolidado

In [64]:
ranking_final = ranking_geral()
display(ranking_final)

if not ranking_final.empty:
    caminho_ranking_final = ARTIFACTS_DIR / "ranking_geral.csv"
    ranking_final.to_csv(caminho_ranking_final, index=False)
    print(f"Ranking consolidado salvo em: {caminho_ranking_final}")

,model_key,modelo,rmse,mae,r2,best_params
0,catboost,CatBoost,438707.387901,251630.093751,0.675836,"{'iterations': 1392, 'learning_rate': 0.173733..."
1,xgboost,XGBoost,458521.719595,255767.143094,0.648200,"{'n_estimators': 703, 'learning_rate': 0.01279..."
2,random_forest,Random Forest,469710.853274,280957.252317,0.627016,"{'n_estimators': 300, 'max_depth': 16, 'min_sa..."
3,lightgbm,LightGBM,471870.128507,287913.522215,0.626413,"{'n_estimators': 1444, 'learning_rate': 0.0134..."
4,ridge,Ridge,522598.972620,320852.327578,0.541375,{'alpha': 10.0}
5,mlp,MLPRegressor,525172.499782,325874.998555,0.536178,"{'width': 128, 'depth': 3, 'activation': 'relu..."
6,lasso,Lasso,544567.887442,348540.647732,0.503654,{'alpha': 10.0}
7,svr,SVR,801496.352402,492965.012962,-0.072870,"{'kernel': 'poly', 'C': 73.58174688831143, 'ep..."


Ranking consolidado salvo em: c:\Users\andriel_orbi\Tiago\git\pipeline-de-modelos-preditores-de-precos-de-imoveis-com-dados-em-tempo-real\modelos de aprendizagem\artifacts_modelagem\ranking_geral.csv


## 17. Rodar todos os modelos

Use esta celula para executar todos os candidatos em sequencia.

In [65]:
for model_key in MODEL_SPECS:
    if model_key in rankings:
        print(f"Pulando {model_key}: ja executado nesta sessao.")
        continue
    treinar_modelo(model_key)

display(ranking_geral())

Pulando xgboost: ja executado nesta sessao.
Pulando lightgbm: ja executado nesta sessao.
Pulando catboost: ja executado nesta sessao.
Pulando random_forest: ja executado nesta sessao.
Pulando ridge: ja executado nesta sessao.
Pulando lasso: ja executado nesta sessao.
Pulando svr: ja executado nesta sessao.
Pulando mlp: ja executado nesta sessao.


,model_key,modelo,rmse,mae,r2,best_params
0,catboost,CatBoost,438707.387901,251630.093751,0.675836,"{'iterations': 1392, 'learning_rate': 0.173733..."
1,xgboost,XGBoost,458521.719595,255767.143094,0.648200,"{'n_estimators': 703, 'learning_rate': 0.01279..."
2,random_forest,Random Forest,469710.853274,280957.252317,0.627016,"{'n_estimators': 300, 'max_depth': 16, 'min_sa..."
3,lightgbm,LightGBM,471870.128507,287913.522215,0.626413,"{'n_estimators': 1444, 'learning_rate': 0.0134..."
4,ridge,Ridge,522598.972620,320852.327578,0.541375,{'alpha': 10.0}
5,mlp,MLPRegressor,525172.499782,325874.998555,0.536178,"{'width': 128, 'depth': 3, 'activation': 'relu..."
6,lasso,Lasso,544567.887442,348540.647732,0.503654,{'alpha': 10.0}
7,svr,SVR,801496.352402,492965.012962,-0.072870,"{'kernel': 'poly', 'C': 73.58174688831143, 'ep..."
